# 06. Official MaintIE split and baseline training data

Recreates the official 80/10/10 split (seed 1337) by running MaintIE's own `create_datasets.py`, maps it to
`gold_release.json`, checks overlap with our prompt pool, and builds the training files for:
- **gold_official**: Mistral trained on the 860 expert-labelled official training texts
- **distilled_860**: Mistral trained on 860 random teacher-labelled texts (quantity-matched control)

In [ ]:
import os, glob, json, subprocess, random
from collections import Counter
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if not os.path.exists("/content/maintie"):
    os.system("git clone -q https://github.com/nlp-tlp/maintie.git /content/maintie")
BASE = "/content/drive/MyDrive/mistral_ft"

res = subprocess.run(["python", "create_datasets.py"], cwd="/content/maintie/models", capture_output=True, text=True)
print(res.stdout[-600:], res.stderr[-400:])

gold = json.load(open("/content/maintie/data/gold_release.json"))
key = lambda toks: " ".join(toks)
gold_by_text = {}
for i, r in enumerate(gold):
    gold_by_text.setdefault(key(r["tokens"]), []).append(i)

In [ ]:
SPLIT_DIR = "/content/maintie/models/data/g-3"   # gold, level 3 (224 types)

def to_gold_indices(path):
    used, out = Counter(), []
    for x in json.load(open(path)):
        k = key(x["tokens"]); ids = gold_by_text.get(k, [])
        if used[k] < len(ids):
            out.append(ids[used[k]]); used[k] += 1
    return out

files = {n: next(p for p in glob.glob(f"{SPLIT_DIR}/*.json") if n in os.path.basename(p).lower())
         for n in ("train", "dev", "test")}
official = {n: to_gold_indices(p) for n, p in files.items()}
splits = json.load(open(f"{BASE}/data/splits.json"))
pool = set(splits["excluded_prompt_pool"])
T = set(official["test"])
print({n: len(v) for n, v in official.items()})
print("official test in prompt pool:", len(T & pool), "| in internal val:", len(T & set(splits["val_gold_indices"])),
      "| in internal test:", len(T & set(splits["test_gold_indices"])))
json.dump({"official_" + n: sorted(v) for n, v in official.items()}, open(f"{BASE}/data/official_split.json", "w"))

In [ ]:
INSTRUCTION = ("Extract the entities (with their MaintIE types) and the relations from this maintenance "
               "work order. Answer with JSON only.\nWork order: ")

def target(rec):
    return json.dumps({
        "entities": [{"text": " ".join(rec["tokens"][e["start"]:e["end"]]), "type": e["type"]} for e in rec["entities"]],
        "relations": [{"head": r["head"], "tail": r["tail"], "type": r["type"]} for r in rec["relations"]],
    }, ensure_ascii=False)

def example(rec):
    return {"messages": [{"role": "user", "content": INSTRUCTION + rec["text"]},
                         {"role": "assistant", "content": target(rec)}]}

off = json.load(open(f"{BASE}/data/official_split.json"))
os.makedirs(f"{BASE}/data_gold", exist_ok=True)
for name, k in [("train", "official_train"), ("val", "official_dev"), ("test", "official_test")]:
    with open(f"{BASE}/data_gold/{name}.jsonl", "w") as f:
        for i in off[k]:
            f.write(json.dumps(example(gold[i]), ensure_ascii=False) + "\n")
print("data_gold written")

In [ ]:
# Quantity-matched control: 860 random teacher-labelled examples, same val/test as data_gold
OUT = f"{BASE}/data_distilled_860"; os.makedirs(OUT, exist_ok=True)
rows = [l for l in open(f"{BASE}/data/train.jsonl") if l.strip()]
silver_ids = splits["train_silver_indices"]
pick = sorted(random.Random(2026).sample(range(len(rows)), 860))
with open(f"{OUT}/train.jsonl", "w") as f:
    for k in pick:
        f.write(rows[k])
for name in ("val", "test"):
    open(f"{OUT}/{name}.jsonl", "w").write(open(f"{BASE}/data_gold/{name}.jsonl").read())
json.dump({"sample_seed": 2026, "train_silver_indices": [silver_ids[k] for k in pick]}, open(f"{OUT}/sample.json", "w"))
print("data_distilled_860 written")

Train both with notebook 04: set `DATA_DIR` to `data_gold` / `data_distilled_860`, `RUN_NAME` to `gold_official_v1` / `distilled_860_v1`, and `num_train_epochs=5` (best checkpoint by validation loss).